# AGENTS026 – Hour 0–1 Notebook (AMD Env Aligned)

This notebook sets up the AMD MI300X environment, configures access to a vLLM server (started from a separate terminal), and creates a base Pydantic AI agent that will be reused for AGENTS026 (incident diagnosis & resolution).

Run the cells in order once per session before working on later notebooks.


## Section 1 – GPU & Workspace Sanity Check

In this section we verify the AMD MI300X GPU is visible and inspect the working directory. This is purely diagnostic and should run quickly.


In [1]:
import os
import subprocess

def run(cmd: str):
    print(f'\n$ {cmd}')
    try:
        out = subprocess.check_output(
            cmd, shell=True, stderr=subprocess.STDOUT, text=True
        )
        print(out)
    except subprocess.CalledProcessError as e:
        print(e.output)

# Basic identity and location
run('whoami')
run('pwd')
run('ls')

# Check AMD GPU status – one of these should work in the AMD notebook image
run('rocm-smi || amd-smi || echo "rocm-smi/amd-smi not found in PATH"')



$ whoami
root


$ pwd
/workspace


$ ls
agents026
agents026_hour0-1_nb.ipynb
build_airbnb_agent_mcp.ipynb
myenv


$ rocm-smi || amd-smi || echo "rocm-smi/amd-smi not found in PATH"


============================================ ROCm System Management Interface ============================================
====================================================== Concise Info ======================================================
Device  Node  IDs              Temp        Power     Partitions          SCLK    MCLK    Fan  Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Junction)  (Socket)  (Mem, Compute, ID)                                                  
0       8     0x74b5,   16815  49.0°C      148.0W    NPS1, SPX, 0        139Mhz  900Mhz  0%   auto  750.0W  89%    0%    
================================================== End of ROCm SMI Log ===================================================



## Section 2 – Launch vLLM Server (from Terminal, Not Notebook)

In the AMD workshop environment, the recommended pattern is to start the vLLM server from a separate terminal, not from inside the notebook itself. This mirrors the Airbnb MCP sample notebook.

1. Open a terminal from the Jupyter UI.
2. Run the following command to start the vLLM server on the AMD MI300X GPU (adjust model if needed):

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
  --served-model-name Qwen3-30B-A3B \
  --api-key abc-123 \
  --port 8000 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --trust-remote-code
```

You may use a smaller model (e.g. `Qwen/Qwen2-1.5B-Instruct`) for faster startup; just keep the `--served-model-name` consistent with the name you will use below.

3. Optionally, in another terminal run:

```bash
watch rocm-smi
```

to monitor GPU utilization while vLLM loads and serves the model.


## Section 3 – Configure Client Environment Variables

Now that the vLLM server is running, configure the notebook to talk to it via the OpenAI-compatible API (same pattern as the AMD Airbnb MCP example).


In [2]:
BASE_URL = 'http://localhost:8000/v1'  # vLLM OpenAI-compatible endpoint
OPENAI_API_KEY = 'abc-123'             # must match the key passed to vLLM

import os
os.environ['BASE_URL'] = BASE_URL
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('Config set:', BASE_URL)


Config set: http://localhost:8000/v1


Quick check: list models served by vLLM using the configured endpoint.


In [3]:
import subprocess
cmd = 'curl ' + BASE_URL + '/models -H "Authorization: Bearer ' + OPENAI_API_KEY + '"'
print('$ ' + cmd)
try:
    out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    print(out)
except subprocess.CalledProcessError as e:
    print('vLLM not reachable – check that the server is running in the terminal.')
    print(e.output)


$ curl http://localhost:8000/v1/models -H "Authorization: Bearer abc-123"
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   466  100   466    0     0   318k      0 --:--:-- --:--:-- --:--:--  455k
{"object":"list","data":[{"id":"Qwen3-30B-A3B","object":"model","created":1781095773,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B","parent":null,"max_model_len":40960,"permission":[{"id":"modelperm-a4df139cfc58d9e9","object":"model_permission","created":1781095773,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}


## Section 4 – Create Project Skeleton

Create a minimal on-disk project structure for the AGENTS026 solution. This will help keep data, agents, and notebooks organized.


In [4]:
from pathlib import Path

root = Path.cwd() / 'agents026'
subdirs = ['data', 'models', 'agents', 'notebooks', 'logs']

for sub in subdirs:
    (root / sub).mkdir(parents=True, exist_ok=True)

print('Created project structure under:', root)
for path in sorted(root.rglob('*')):
    print('-', path.relative_to(Path.cwd()))


Created project structure under: /workspace/agents026
- agents026/agents
- agents026/data
- agents026/logs
- agents026/models
- agents026/notebooks


## Section 5 – Install Python Dependencies

The AMD base image already includes ROCm and PyTorch. Here we install the higher-level libraries needed for AGENTS026:

- `pydantic-ai-slim` and `openai` for agent framework and OpenAI-compatible client.
- `pandas`, `numpy`, `matplotlib` for data handling and plotting.
- `faiss-cpu` for a simple vector index (incident similarity search).

Use `%pip` so the environment is correctly wired to the active kernel.


In [5]:
%pip install -q \
    pydantic-ai-slim \
    openai \
    pandas \
    numpy \
    matplotlib \
    faiss-cpu

print('Installed: pydantic-ai-slim, openai, pandas, numpy, matplotlib, faiss-cpu')


Note: you may need to restart the kernel to use updated packages.
Installed: pydantic-ai-slim, openai, pandas, numpy, matplotlib, faiss-cpu


## Section 6 – Define OpenAI-Compatible Model & Base Agent (Pydantic AI)

With vLLM serving our Qwen model and dependencies installed, we now:

1. Create an `OpenAIProvider` pointing at the vLLM endpoint.
2. Wrap it in an `OpenAIChatModel` for Pydantic AI.
3. Instantiate a simple `Agent` that will be the basis for later AGENTS026 agents (RCA, remediation planning, etc.).


In [6]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent

base_url = os.environ['BASE_URL']
api_key = os.environ['OPENAI_API_KEY']

provider = OpenAIProvider(
    base_url=base_url,
    api_key=api_key,
)

# Use the same served-model-name you passed to vLLM in the terminal
MODEL_NAME = 'Qwen3-30B-A3B'  # or e.g. 'Qwen2-1.5B-Instruct' if you used that

agent_model = OpenAIChatModel(MODEL_NAME, provider=provider)

# Generic LLM agent; later we’ll specialize it into RCA / remediation agents
base_agent = Agent(
    model=agent_model,
)

print('Base agent initialized with model:', MODEL_NAME)


Base agent initialized with model: Qwen3-30B-A3B


## Section 7 – Async Helper and Smoke Test

Define a small async helper to run the base agent and confirm everything is wired correctly. This mirrors the pattern used in the Airbnb MCP workshop notebook.


In [9]:
import asyncio

async def run_llm(prompt: str) -> str:
    # For now we don’t use MCP servers; we’ll add tools/agents later
    result = await base_agent.run(prompt)
    return result.output

# Test the agent with a simple question
response = await run_llm('In one sentence, explain what an SRE does.')
print('Model reply:', response)


Model reply: 

An SRE (Site Reliability Engineer) ensures system reliability and availability by combining software engineering and operations practices to automate, monitor, and optimize infrastructure while balancing innovation with stability.


If you see a coherent answer, your Hour 0–1 setup is complete:

- AMD GPU is visible.
- vLLM is serving a Qwen model on `localhost:8000`.
- Environment variables and OpenAI-compatible client are configured.
- Pydantic AI is installed and can talk to the model through `base_agent`.

In the next notebooks, you will build the AGENTS026 incident detection, similarity search, RCA, and remediation agents on top of this base.
